# Reproducing the LLM psychedelic-simulation paper

This notebook implements the **logic of the uploaded draft** as a class-based, Colab-ready pipeline.

## What this notebook does
- loads an **approved Erowid export** supplied by you
- generates neutral and psychedelic narratives with **real API calls**
- scores each narrative with a **30-item MEQ-style rubric**
- computes **SBERT similarity** to the human reference corpus
- recreates the paper's **four figures**
- exports **CSV tables** for the reported analyses

## Important honesty notes
The draft is not fully specified. This notebook is therefore **configurable** rather than pretending to be an exact turnkey replication. The biggest gaps are:
1. the draft contains **conflicting model rosters**
2. the exact **supplementary prompts** were not included
3. the exact **approved Erowid export** was not included
4. the exact **SBERT checkpoint** was not specified
5. the draft flips between **NLP similarity** and **NLP distance**

The pipeline keeps all of those as explicit configuration choices.


In [ ]:
!pip -q install -U openai anthropic google-genai huggingface_hub sentence-transformers scikit-learn scipy statsmodels pandas numpy matplotlib tqdm

In [ ]:
from pathlib import Path
import json
import shutil
import sys

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

template_path = PROJECT_ROOT / "api_keys.template.json"
api_keys_path = PROJECT_ROOT / "api_keys.json"

if template_path.exists() and not api_keys_path.exists():
    shutil.copy(template_path, api_keys_path)
    print(f"Created {api_keys_path.name} from the template. Fill in the keys before running the experiment.")
else:
    print("api_keys.json already exists or api_keys.template.json is missing.")


## Configure the experiment

Choose which paper variant you want to reproduce by default:

- `methods_results` → Gemini + Claude + ChatGPT + Llama + Falcon
- `abstract` → GPT-4o + Claude 3.5 Sonnet + Llama-3.1-70B

Because the draft is inconsistent, both are supported.


In [ ]:
from llm_psychedelic_repro.config import build_experiment_config, ProviderSecrets

PAPER_VARIANT = "methods_results"   # or: "abstract"
SMOKE_TEST = True                   # set False for the full 100 samples per cell
FORCE_REGENERATE = False            # set True to ignore cached CSV outputs

config = build_experiment_config(PAPER_VARIANT)
config.paths.project_root = PROJECT_ROOT
config.paths.data_dir = PROJECT_ROOT / "data"
config.paths.input_dir = PROJECT_ROOT / "data" / "input"
config.paths.cache_dir = PROJECT_ROOT / "data" / "cache"
config.paths.output_dir = PROJECT_ROOT / "outputs"
config.paths.figures_dir = PROJECT_ROOT / "outputs" / "figures"
config.paths.tables_dir = PROJECT_ROOT / "outputs" / "tables"
config.paths.logs_dir = PROJECT_ROOT / "outputs" / "logs"
config.existing_generations_path = PROJECT_ROOT / "outputs" / "generated_reports.csv"
config.data.approved_export_path = PROJECT_ROOT / "data" / "input" / "erowid_approved_export.csv"

if SMOKE_TEST:
    config.generation.samples_per_cell = 2

secrets = ProviderSecrets.load(api_keys_path)
config.paths.ensure()

print("Enabled models:")
for m in config.enabled_models():
    print(f" - {m.display_name} | provider={m.provider} | model_name={m.model_name}")

print("\nHuman reference corpus path:", config.data.approved_export_path)
print("Samples per model x condition:", config.generation.samples_per_cell)


## Expected human-corpus file format

Place your approved Erowid export at:

`data/input/erowid_approved_export.csv`

Minimum required columns:
- `report_text`
- `substance`

Optional columns:
- `co_substances`
- `title`

The loader will:
- normalize substance names
- keep only the 5 target psychedelic substances
- keep only single-substance reports
- remove duplicates
- require at least 200 characters of text


In [ ]:
from llm_psychedelic_repro.pipeline import PsychedelicPaperPipeline

pipeline = PsychedelicPaperPipeline(config=config, secrets=secrets)

# Run the full pipeline.
# This performs:
# 1. load human corpus
# 2. generate narratives
# 3. MEQ scoring
# 4. SBERT similarity
# 5. statistics + figures + tables
results = pipeline.run_all(
    force_generation=FORCE_REGENERATE,
    force_meq=FORCE_REGENERATE,
    force_similarity=FORCE_REGENERATE,
)

print("Done.")
print("Scored rows:", len(results["scored_df"]))


In [ ]:
# Preview the scored dataset
results["scored_df"].head()


In [ ]:
# Show the generated figures
from IPython.display import Image, display
for key, path in results["figures"].items():
    print(key, path)
    display(Image(filename=str(path)))


In [ ]:
# Inspect table names and a few rows from each
for name, table in results["tables"].items():
    print("\n====", name, "====")
    display(table.head())


## Notes on exact replication

This notebook reproduces the **paper's computational logic** but cannot guarantee the exact reported numeric results unless you also match:

- the exact approved Erowid export used by the authors
- the exact hidden supplementary prompts
- the exact model snapshots / provider endpoints
- the exact SBERT checkpoint
- any post-hoc exclusion rules not fully documented in the draft
